In [1]:
import wandb
import pandas as pd

In [2]:
api = wandb.Api()
runs = api.runs(path="harvardml/opt-olmo")

# Comments

This gets a df with all jobs that finished (i.e. did not diverge or die)

Checkpoints are at /n/holyscratch01/barak_lab/Lab/opt-olmo/ckpts and the names are given in the "name" column of the df

In [10]:

df = pd.DataFrame()

steps = [100, 300, 500, 900, 1700, 2900, 4900, 8500, 14600, 25000]
step = 25000
groups = ["optimizers-1", "optimizers-2"]

for run in runs:
    if run.group in groups:
        history = run.history(samples=25000, keys=["eval/c4_val/CrossEntropyLoss"])
        if "_step" in history and step in list(history["_step"]): # Check if job finished
            run_dict = {}
            #run_dict["id"] = run.id
            run_dict["group"] = run.group
            run_dict["optimizer"] = run.config["optimizer"]["name"]
            if "neuron_only" in run.config["optimizer"] and run.config["optimizer"]["neuron_only"]:
                run_dict["optimizer"] += " (neuron only)"
            if "att_correction" in run.config["optimizer"] and run.config["optimizer"]["att_correction"]:
                run_dict["optimizer"] += " (att correction)"
            run_dict["learning_rate"] = run.config["optimizer"]["learning_rate"]
            run_dict["val_loss"] = list(history[history["_step"] == step]["eval/c4_val/CrossEntropyLoss"])[0]
            run_dict["name"] = run.name.replace("olmo_", "")
            run_df = pd.DataFrame(run_dict, index=[0])
            df = pd.concat([df, run_df])

In [13]:
df.head()

,group,optimizer,learning_rate,val_loss,name
0,optimizers-1,signsgd,0.01000,3.162558,25408809_4
0,optimizers-1,signsgd,0.00001,3.580334,25408809_1
0,optimizers-1,signsgd,0.00100,3.083332,25408809_3
0,optimizers-1,signsgd,0.00010,3.140393,25408809_2
0,optimizers-1,signsgd,0.00316,3.137564,25388222_6


In [15]:
len(df)

55